# 02_Auditoria_Dataset.ipynb
## Auditoría del conjunto de imágenes — Banana_Guava_Classifier

**Objetivo:** inspeccionar el ZIP original sin modificarlo, extraer las etiquetas desde los nombres, verificar que las imágenes puedan abrirse, revisar dimensiones, calcular hashes SHA-256 para detectar duplicados exactos y generar reportes CSV.

### Reglas de esta auditoría
- La carpeta original se conserva intacta. Para este repositorio, coloca una copia de las imágenes en `data/raw/`.
- No se eliminan, renombran ni sobrescriben imágenes.
- Un duplicado exacto se determina comparando el SHA-256 del contenido del archivo.
- Una coincidencia de hash con etiquetas diferentes se marca como **posible conflicto de etiquetado**; no se corrige automáticamente.
- Esta auditoría no determina si una fruta está correctamente etiquetada visualmente. Eso requiere revisión humana.

## 1. Instalar e importar dependencias

Si ya tienes instaladas las librerías, puedes ejecutar la celda sin problema. En Jupyter, `%pip` instala en el entorno del notebook.

In [ ]:
%pip install -q pandas pillow matplotlib

In [ ]:
from pathlib import Path
from collections import defaultdict
from itertools import combinations
from io import BytesIO
import hashlib
import re
import zipfile
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)

print("Dependencias cargadas correctamente.")

## 2. Configurar la carpeta del dataset

El notebook buscará las imágenes directamente en la carpeta indicada en Windows. No es necesario comprimirlas en un ZIP.

La búsqueda es recursiva: también encontrará imágenes dentro de subcarpetas de `evaluacion_imagenes`.

In [ ]:
from pathlib import Path

# Ruta principal en la computadora del usuario
PROJECT_ROOT = Path.cwd()

# Coloca las imágenes originales dentro de data/raw/ (sin cambiar sus nombres).
RUTA_DATASET = PROJECT_ROOT / "data" / "raw"

# Los reportes de auditoría se guardarán dentro del proyecto.
CARPETA_SALIDA = PROJECT_ROOT / "results" / "auditoria"

if not RUTA_DATASET.exists():
    raise FileNotFoundError(
        "No se encontró la carpeta del dataset. Verifica esta ruta:\\n"
        f"{RUTA_DATASET}"
    )

if not RUTA_DATASET.is_dir():
    raise NotADirectoryError(f"La ruta no es una carpeta: {RUTA_DATASET}")

print("Dataset localizado correctamente:")
print(RUTA_DATASET)
print("Los reportes se guardarán en:")
print(CARPETA_SALIDA)

## 3. Definir el formato esperado de los nombres

Se espera un nombre con esta estructura:

`Banana__Class_A__image_0.jpg`

Las clases reconocidas serán:
- Fruta: `Banana` o `Guava`
- Calidad: `Class_A`, `Class_B` o `Defect`

Los archivos que no cumplan el patrón se conservan en el inventario, pero se marcan como nombre no reconocido.

In [ ]:
patron_nombre = re.compile(
    r"^(Banana|Guava)__(Class_A|Class_B|Defect)__image_(\d+)\.(jpg|jpeg|png)$",
    flags=re.IGNORECASE
)

def extraer_etiquetas(nombre_archivo):
    coincidencia = patron_nombre.match(Path(nombre_archivo).name)
    if not coincidencia:
        return {
            "fruta": None,
            "calidad": None,
            "indice_archivo": None,
            "nombre_reconocido": False
        }

    fruta, calidad, indice, extension = coincidencia.groups()
    fruta = {"banana": "Banana", "guava": "Guava"}[fruta.lower()]
    calidad = {
        "class_a": "Class_A",
        "class_b": "Class_B",
        "defect": "Defect"
    }[calidad.lower()]

    return {
        "fruta": fruta,
        "calidad": calidad,
        "indice_archivo": int(indice),
        "nombre_reconocido": True
    }

print(extraer_etiquetas("Banana__Class_A__image_0.jpg"))

## 4. Inspeccionar las imágenes de la carpeta

Esta celda:
1. Busca imágenes JPG, JPEG y PNG dentro de `RUTA_DATASET` y sus subcarpetas.
2. Calcula SHA-256.
3. Intenta verificar y abrir cada imagen.
4. Registra formato, dimensiones, modo de color y tamaño.
5. No modifica, renombra ni elimina archivos.

In [ ]:
extensiones_validas = {".jpg", ".jpeg", ".png"}

archivos_imagenes = sorted(
    archivo for archivo in RUTA_DATASET.rglob("*")
    if archivo.is_file()
    and archivo.suffix.lower() in extensiones_validas
    and not archivo.name.startswith("._")
    and not archivo.name.startswith(".")
)

registros = []

for numero, ruta_imagen in enumerate(archivos_imagenes, start=1):
    nombre = ruta_imagen.name
    etiquetas = extraer_etiquetas(nombre)

    try:
        datos = ruta_imagen.read_bytes()
        hash_sha256 = hashlib.sha256(datos).hexdigest()
    except OSError as error:
        datos = b""
        hash_sha256 = None

    registro = {
        "ruta_relativa": str(ruta_imagen.relative_to(RUTA_DATASET)),
        "ruta_completa": str(ruta_imagen),
        "nombre_archivo": nombre,
        "extension": ruta_imagen.suffix.lower(),
        "tamano_bytes": len(datos),
        "sha256": hash_sha256,
        **etiquetas,
        "imagen_valida": False,
        "formato_imagen": None,
        "ancho": None,
        "alto": None,
        "modo_color": None,
        "error_lectura": None
    }

    if datos:
        try:
            with Image.open(BytesIO(datos)) as imagen:
                imagen.verify()

            with Image.open(BytesIO(datos)) as imagen:
                registro["imagen_valida"] = True
                registro["formato_imagen"] = imagen.format
                registro["ancho"], registro["alto"] = imagen.size
                registro["modo_color"] = imagen.mode

        except (UnidentifiedImageError, OSError, ValueError) as error:
            registro["error_lectura"] = str(error)
    else:
        registro["error_lectura"] = "No se pudo leer el archivo."

    registros.append(registro)

    if numero % 100 == 0:
        print(f"Revisadas {numero:,} de {len(archivos_imagenes):,} imágenes...")

df = pd.DataFrame(registros)

print("\\nAuditoría de archivos terminada.")
print("Imágenes encontradas:", len(df))
display(df.head())

## 5. Resumen general de integridad

Se revisan cantidad de archivos, imágenes válidas, archivos con nombres no reconocidos, extensiones y dimensiones.

In [ ]:
resumen_integridad = pd.DataFrame([
    ["Archivos inspeccionados", len(df)],
    ["Imágenes válidas", int(df["imagen_valida"].sum())],
    ["Imágenes no válidas", int((~df["imagen_valida"]).sum())],
    ["Nombres reconocidos", int(df["nombre_reconocido"].sum())],
    ["Nombres no reconocidos", int((~df["nombre_reconocido"]).sum())],
    ["Hashes SHA-256 distintos", int(df["sha256"].nunique())],
], columns=["indicador", "cantidad"])

display(resumen_integridad)

print("Extensiones:")
display(df["extension"].value_counts(dropna=False).rename_axis("extension").to_frame("cantidad"))

print("Dimensiones más frecuentes:")
display(
    df.groupby(["ancho", "alto"], dropna=False)
      .size()
      .reset_index(name="cantidad")
      .sort_values("cantidad", ascending=False)
      .head(15)
)

## 6. Distribución por fruta y calidad

La tabla se calcula a partir de las etiquetas extraídas de los nombres de archivo. Los nombres no reconocidos se reportan aparte y no se asignan a una clase por inferencia.

In [ ]:
df_etiquetados = df[df["nombre_reconocido"]].copy()

resumen_clases = (
    df_etiquetados
    .groupby(["fruta", "calidad"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=["Banana", "Guava"], columns=["Class_A", "Class_B", "Defect"], fill_value=0)
)

resumen_clases["Total"] = resumen_clases.sum(axis=1)
resumen_clases.loc["Total"] = resumen_clases.sum(axis=0)

display(resumen_clases)

ax = (
    df_etiquetados.groupby(["fruta", "calidad"])
    .size().unstack(fill_value=0)
    .reindex(index=["Banana", "Guava"], columns=["Class_A", "Class_B", "Defect"], fill_value=0)
    .plot(kind="bar", figsize=(9, 5))
)
ax.set_title("Distribución de imágenes por fruta y calidad")
ax.set_xlabel("Fruta")
ax.set_ylabel("Número de imágenes")
ax.legend(title="Calidad")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Detectar duplicados exactos

Se agrupan los archivos que tienen el mismo SHA-256. Un grupo con dos o más rutas representa contenido binario idéntico.

Se generan dos reportes:
- **Grupos de duplicados:** un registro por hash repetido.
- **Pares de duplicados:** todas las parejas dentro de cada grupo, útiles para inspección.

In [ ]:
grupos_hash = (
    df.groupby("sha256")
      .filter(lambda grupo: len(grupo) > 1)
      .sort_values(["sha256", "ruta_relativa"])
)

filas_grupos = []
filas_pares = []

for hash_valor, grupo in grupos_hash.groupby("sha256"):
    rutas = grupo["ruta_relativa"].tolist()
    etiquetas = sorted({
        f"{f} / {c}" for f, c in zip(grupo["fruta"], grupo["calidad"])
        if pd.notna(f) and pd.notna(c)
    })
    filas_grupos.append({
        "sha256": hash_valor,
        "cantidad_archivos": len(grupo),
        "etiquetas_encontradas": " | ".join(etiquetas),
        "posible_conflicto_etiquetas": len(etiquetas) > 1,
        "rutas": " || ".join(rutas)
    })

    for ruta_a, ruta_b in combinations(rutas, 2):
        fila_a = grupo.loc[grupo["ruta_relativa"] == ruta_a].iloc[0]
        fila_b = grupo.loc[grupo["ruta_relativa"] == ruta_b].iloc[0]
        etiqueta_a = f"{fila_a['fruta']} / {fila_a['calidad']}"
        etiqueta_b = f"{fila_b['fruta']} / {fila_b['calidad']}"
        filas_pares.append({
            "sha256": hash_valor,
            "ruta_1": ruta_a,
            "etiqueta_1": etiqueta_a,
            "ruta_2": ruta_b,
            "etiqueta_2": etiqueta_b,
            "etiquetas_diferentes": etiqueta_a != etiqueta_b
        })

df_grupos_duplicados = pd.DataFrame(filas_grupos)
df_pares_duplicados = pd.DataFrame(filas_pares)

print("Grupos de contenido duplicado:", len(df_grupos_duplicados))
print("Archivos que pertenecen a grupos duplicados:", len(grupos_hash))
print("Pares de archivos idénticos:", len(df_pares_duplicados))

if not df_grupos_duplicados.empty:
    display(df_grupos_duplicados)
else:
    print("No se encontraron duplicados exactos.")

if not df_pares_duplicados.empty:
    print("Pares con etiquetas diferentes:")
    display(df_pares_duplicados[df_pares_duplicados["etiquetas_diferentes"]])

## 8. Revisar posibles conflictos de etiquetado

Un conflicto se presenta cuando el mismo contenido (mismo SHA-256) aparece asignado a más de una combinación de fruta y calidad.

**Importante:** el notebook solo señala el caso. No decide cuál etiqueta es correcta ni elimina archivos.

In [ ]:
if not df_grupos_duplicados.empty:
    conflictos = df_grupos_duplicados[
        df_grupos_duplicados["posible_conflicto_etiquetas"]
    ].copy()
else:
    conflictos = pd.DataFrame(columns=df_grupos_duplicados.columns)

print("Grupos duplicados con más de una etiqueta:", len(conflictos))
display(conflictos)

## 9. Crear un inventario de revisión manual

El inventario incluye una columna `revision_manual` para que el equipo documente si la etiqueta debe conservarse, corregirse o quedar pendiente. El notebook no modifica los nombres ni los archivos originales.

In [ ]:
df_inventario = df.copy()
df_inventario["revision_manual"] = ""
df_inventario["etiqueta_verificada"] = ""
df_inventario["observaciones"] = ""

# Marcar automáticamente los archivos que requieren atención, sin corregirlos.
hashes_conflictivos = set(conflictos["sha256"]) if not conflictos.empty else set()
df_inventario["requiere_revision"] = (
    (~df_inventario["nombre_reconocido"])
    | (~df_inventario["imagen_valida"])
    | (df_inventario["sha256"].isin(hashes_conflictivos))
)

display(
    df_inventario[
        df_inventario["requiere_revision"]
    ][[
        "ruta_relativa", "fruta", "calidad", "imagen_valida",
        "sha256", "revision_manual", "etiqueta_verificada", "observaciones"
    ]].head(100)
)

## 10. Exportar los reportes

Se crea una carpeta `auditoria_resultados` en el Escritorio indicado. Los CSV pueden abrirse en Excel y utilizarse posteriormente para preparar los conjuntos de entrenamiento.

In [ ]:
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

df_inventario.to_csv(
    CARPETA_SALIDA / "inventario_imagenes.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_clases.to_csv(
    CARPETA_SALIDA / "resumen_fruta_calidad.csv",
    encoding="utf-8-sig"
)

df_grupos_duplicados.to_csv(
    CARPETA_SALIDA / "grupos_duplicados_exactos.csv",
    index=False,
    encoding="utf-8-sig"
)

df_pares_duplicados.to_csv(
    CARPETA_SALIDA / "pares_duplicados_exactos.csv",
    index=False,
    encoding="utf-8-sig"
)

conflictos.to_csv(
    CARPETA_SALIDA / "posibles_conflictos_etiquetado.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_integridad.to_csv(
    CARPETA_SALIDA / "resumen_integridad.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Reportes guardados en:", CARPETA_SALIDA.resolve())
for archivo in sorted(CARPETA_SALIDA.iterdir()):
    print("-", archivo.name)

## 11. Conclusión automática de la auditoría

Esta celda resume los indicadores principales. La decisión de corregir etiquetas debe tomarse después de revisar los casos señalados y, cuando sea posible, consultar las imágenes.

In [ ]:
total = len(df)
validas = int(df["imagen_valida"].sum())
no_reconocidas = int((~df["nombre_reconocido"]).sum())
grupos_dup = len(df_grupos_duplicados)
conflictos_n = len(conflictos)

print("========== RESULTADO DE AUDITORÍA ==========")
print(f"Archivos inspeccionados: {total}")
print(f"Imágenes válidas: {validas}")
print(f"Imágenes no válidas: {total - validas}")
print(f"Nombres no reconocidos: {no_reconocidas}")
print(f"Grupos de duplicados exactos: {grupos_dup}")
print(f"Grupos con posibles conflictos de etiqueta: {conflictos_n}")

if total == 1310:
    print("El número total coincide con las 1,310 imágenes reportadas previamente.")
else:
    print("ATENCIÓN: el total difiere de las 1,310 imágenes reportadas previamente.")

if conflictos_n or no_reconocidas or validas != total:
    print("ESTADO: requiere revisión antes de preparar los conjuntos de entrenamiento.")
else:
    print("ESTADO: no se detectaron problemas estructurales en estas verificaciones.")

print("\nNota: una auditoría automática no sustituye la validación visual de las etiquetas.")

## 12. Criterios para avanzar al entrenamiento

Antes de crear `train`, `validation` y `test`, debemos:

1. Revisar los duplicados exactos y resolver las etiquetas contradictorias.
2. Decidir qué hacer con imágenes ilegibles o nombres no reconocidos.
3. Mantener juntas en una misma partición las imágenes duplicadas o casi duplicadas.
4. Revisar la baja representación de `Guava / Class_B`.
5. Congelar una versión auditada del inventario para que los experimentos sean reproducibles.

**No se debe balancear ni duplicar artificialmente el conjunto de prueba.** Cualquier aumento de datos se aplicará únicamente al entrenamiento.

**Ruta configurada:** `C:\Users\sergi\OneDrive\Desktop\evaluacion_imagenes`.
